In [1]:
import paramiko
import stat
from pathlib import PurePosixPath
import os

In [38]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
HOST = os.environ.get("SFTP_HOST")
USER = os.environ.get("SFTP_USER")
PASSWORD = os.environ.get("SFTP_PASSWORD")
PORT = os.environ.get("SFTP_PORT")

#del os.environ["SSH_AUTH_SOCK"]

In [3]:
#import logging
#logging.basicConfig(level=logging.DEBUG)
#paramiko.util.log_to_file("paramiko.log")

In [4]:
def sftp_client(host=HOST, username=USER, port=PORT, password=PASSWORD):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    old_sock = os.environ.pop("SSH_AUTH_SOCK", None)
    connect_kwargs = {
        "username": username,
        "password": password,
        "timeout": 30,
        "allow_agent": False,
        "look_for_keys": False,   # <-- critical: stops ~/.ssh scan
    }
    if port:
        connect_kwargs["port"] = port

    client.connect(host, **connect_kwargs)
    connection = client.open_sftp()

    if old_sock:
        os.environ["SSH_AUTH_SOCK"] = old_sock

    return connection

In [23]:
sftp = sftp_client()

In [24]:
entries = sftp.listdir_attr("/")

In [25]:
len(entries)

132

In [26]:
entries[0].filename

'.quarantine'

In [27]:
def sftp_walk(sftp, root="/", max_depth=10):
    inventory = []

    def _walk(path, depth):
        if depth > max_depth:
            print(f"Max depth reached: {path}")
            return
        try:
            entries = sftp.listdir_attr(path)
        except IOError as e:
            print(f"Skipping {path}: {e}")
            return
        if depth == 1:
            print(f"Visiting {path}...")
        for entry in entries:
            full_path = f"{path.rstrip('/')}/{entry.filename}"
            is_dir = stat.S_ISDIR(entry.st_mode)
            inventory.append({
                "path": full_path,
                "type": "dir" if is_dir else "file",
                "size": entry.st_size,
                "mtime": entry.st_mtime,
            })
            if is_dir:
                _walk(full_path, depth + 1)

    _walk(root, 0)
    return inventory

In [28]:
inv = sftp_walk(sftp, max_depth=100)

Visiting /.quarantine...
Visiting /.tmb...
Visiting /Aleksandrovka...
Visiting /Bar...
Visiting /BerdichevMakhnov_district...
Visiting /Bratslav_county...
Visiting /Bukovina...
Visiting /Byshev...
Visiting /CDIAK archive - Kyiv...
Visiting /Cherkassy_district...
Visiting /Chernigov...
Visiting /Chernobyl...
Visiting /Chigirin_district...
Visiting /Chortkiv district...
Visiting /Crimea...
Visiting /DAHEO...
Visiting /DAKO...
Visiting /DAKO_280...
Visiting /DAKO_384-10_1897census_Skvira_district...
Visiting /DAKO_384-11_1897census_Tarashcha_district...
Visiting /DAKO_384-12_1897census_Uman_district...
Visiting /DAKO_384-15_1897census_Berdichev_district...
Visiting /DAKO_384-2_1897census_Kyiv_city...
Visiting /DAKO_384-3_1897census_Kyiv_district...
Visiting /DAKO_384-4_1897census_Berdichev_district...
Visiting /DAKO_384-5_1897census_Vasilkov_district...
Visiting /DAKO_384-6_1897census_Zvenigorodka_district...
Visiting /DAKO_384-7_1897census_Kanev-district...
Visiting /DAKO_384-8_1897censu

In [29]:
len(inv)

428855

In [30]:
inv[:10]

[{'path': '/.quarantine', 'type': 'dir', 'size': None, 'mtime': 1775123291},
 {'path': '/.tmb', 'type': 'dir', 'size': None, 'mtime': 1573220106},
 {'path': '/Aleksandrovka', 'type': 'dir', 'size': None, 'mtime': 1752861839},
 {'path': '/Aleksandrovka/12-2-167.pdf',
  'type': 'file',
  'size': 277905450,
  'mtime': 1744627287},
 {'path': '/Aleksandrovka/12-2-233.pdf',
  'type': 'file',
  'size': 257001276,
  'mtime': 1744627207},
 {'path': '/Aleksandrovka/12-2-269.pdf',
  'type': 'file',
  'size': 288342445,
  'mtime': 1744142601},
 {'path': '/Aleksandrovka/12-2-275.pdf',
  'type': 'file',
  'size': 193062022,
  'mtime': 1744627183},
 {'path': '/Aleksandrovka/12-2-280.pdf',
  'type': 'file',
  'size': 616239442,
  'mtime': 1743592635},
 {'path': '/Aleksandrovka/12-2-281.pdf',
  'type': 'file',
  'size': 536665675,
  'mtime': 1743763429},
 {'path': '/Aleksandrovka/12-2-57.pdf',
  'type': 'file',
  'size': 114752569,
  'mtime': 1744627251}]

In [35]:
os.environ["BIRDDOG_NOCODB_ENV"] = "LOCAL"

In [39]:
from birddog.database import Database

2026-08-26 11:11:20,086 [INFO] Using LOCAL nocodb api: http://localhost:8080


In [40]:
import json
with open("ftp_snapshot.json", "w") as file:
    file.write(json.dumps(inv))

In [59]:
def write_sftp_inventory_csv(inventory, output_path):
    if not inventory:
        raise ValueError("Inventory is empty")
    
    fields = ["path", "filename", "suffix", "depth", "type", "size", "mtime"]
    
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fields,
            delimiter=",",
            quoting=csv.QUOTE_MINIMAL,
        )
        writer.writeheader()
        for item in inventory:
            p = PurePosixPath(item["path"].lstrip("/"))
            is_dir = item["type"] == "dir"
            row = {
                "path": str(p.parent) if str(p.parent) != "." else "",
                "filename": p.name,
                "suffix": "" if is_dir else p.suffix.lstrip("."),
                "depth": len(p.parts) - 1,
                "type": item["type"],
                "size": item["size"],
                "mtime": datetime.fromtimestamp(item["mtime"]).strftime("%Y-%m-%d %H:%M:%S")
                         if isinstance(item["mtime"], (int, float)) else item["mtime"],
            }
            writer.writerow(row)
    
    print(f"Wrote {len(inventory)} rows to {output_path}")

In [60]:
write_sftp_inventory_csv(inv, "research/ftp/ftp_inventory.csv")

Wrote 428855 rows to research/ftp/ftp_inventory.csv


In [50]:
db = Database()

2026-08-26 13:25:06,728 [INFO] creating NocoDBDatabase(host=http://localhost:8080, base_id=p79fvr9cjqgpv5n) instance
2026-08-26 13:25:06,846 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight  tgt_rps   budget_s
  ------------------------------------------------------------------------------------------------------
  localhost:api                          5.00     8.57     7.00       0.00            4        -          -
